In [ ]:
!nvidia-smi

Sat Nov  1 18:55:12 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install tensorflow

In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
# ===== Cell 1: Install & Imports =====
# Run this cell first (may take a minute)
!pip install -q -U "tensorflow>=2.10" gradio efficientnet tensorflow-addons

import os, datetime, pathlib
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, callbacks, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing import image as keras_image
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)


ERROR: Could not find a version that satisfies the requirement tensorflow-addons (from versions: none)
ERROR: No matching distribution found for tensorflow-addons
TensorFlow: 2.19.0


In [5]:
# ===== Cell 2: Mount Google Drive & dataset paths =====
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE = '/content/drive/MyDrive/underwater_plastics'
TRAIN_DIR = os.path.join(BASE, 'train')
VALID_DIR = os.path.join(BASE, 'valid')
TEST_DIR  = os.path.join(BASE, 'test')

print("Train exists:", os.path.exists(TRAIN_DIR))
print("Valid exists:", os.path.exists(VALID_DIR))
print("Test  exists:", os.path.exists(TEST_DIR))


Mounted at /content/drive
Train exists: True
Valid exists: True
Test  exists: True


In [6]:
# ===== Cell 3: Auto-detect classes (expecting 3) =====
train_path = pathlib.Path(TRAIN_DIR)
if not train_path.exists():
    raise FileNotFoundError(f"Train folder not found: {TRAIN_DIR}")

CLASS_NAMES = sorted([p.name for p in train_path.iterdir() if p.is_dir()])
NUM_CLASSES = len(CLASS_NAMES)
print("Detected class folders (train):", CLASS_NAMES)
if NUM_CLASSES != 3:
    print("WARNING: Detected", NUM_CLASSES, "classes. This notebook was built for 3 classes. Proceeding with detected classes.")


Detected class folders (train): ['images', 'labels']


In [7]:
# ===== Cell 4: Create tf.data datasets =====
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
seed = 42

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    labels='inferred',
    label_mode='categorical',
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    shuffle=True,
    seed=seed
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    VALID_DIR,
    labels='inferred',
    label_mode='categorical',
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    shuffle=False
)

test_ds = None
if os.path.exists(TEST_DIR) and len(os.listdir(TEST_DIR))>0:
    try:
        test_ds = tf.keras.preprocessing.image_dataset_from_directory(
            TEST_DIR,
            labels='inferred',
            label_mode='categorical',
            batch_size=BATCH_SIZE,
            image_size=IMG_SIZE,
            shuffle=False
        )
    except Exception as e:
        print("Could not load test dataset:", e)
        test_ds = None

# Prefetch for performance
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)
if test_ds is not None:
    test_ds = test_ds.cache().prefetch(AUTOTUNE)

print("Datasets ready.")


Found 3628 files belonging to 2 classes.
Found 1001 files belonging to 2 classes.
Found 501 files belonging to 2 classes.
Datasets ready.


In [8]:
# ===== Cell 5: Data augmentation + preprocessing =====
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.06),
    layers.RandomTranslation(0.03, 0.03),
    layers.RandomContrast(0.08),
], name="data_augmentation")

preprocess_input = tf.keras.applications.efficientnet.preprocess_input


In [9]:
# ===== Cell 6: Build model (EfficientNetB0 backbone) =====
base_model = EfficientNetB0(include_top=False, weights='imagenet', input_shape=IMG_SIZE + (3,))
base_model.trainable = False

inputs = layers.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.35)(x)
x = layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs, outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,214,309 (16.08 MB)

 Trainable params: 164,482 (642.51 KB)

 Non-trainable params: 4,049,827 (15.45 MB)

In [10]:
# ===== Cell 7: Callbacks and model folder =====
TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
MODEL_DIR = os.path.join(BASE, "models")
os.makedirs(MODEL_DIR, exist_ok=True)
BEST_CHECKPOINT = os.path.join(MODEL_DIR, f"best_model_{TIMESTAMP}.h5")
FINAL_MODEL_DIR = os.path.join(MODEL_DIR, f"final_saved_model_{TIMESTAMP}")

cb_early = callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)
cb_plateau = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
cb_ckpt = callbacks.ModelCheckpoint(BEST_CHECKPOINT, monitor='val_loss', save_best_only=True, verbose=1)

print("Best checkpoint path:", BEST_CHECKPOINT)


Best checkpoint path: /content/drive/MyDrive/underwater_plastics/models/best_model_20251102-065407.h5


In [11]:
# ===== Cell 8: Train stage 1 (head) =====
EPOCHS_HEAD = 12
history1 = model.fit(
    train_ds,
    epochs=EPOCHS_HEAD,
    validation_data=val_ds,
    callbacks=[cb_early, cb_plateau, cb_ckpt]
)


Epoch 1/12
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 921ms/step - accuracy: 0.6161 - loss: 0.7954
Epoch 1: val_loss improved from inf to 0.16132, saving model to /content/drive/MyDrive/underwater_plastics/models/best_model_20251102-065407.h5


114/114 ━━━━━━━━━━━━━━━━━━━━ 352s 3s/step - accuracy: 0.6171 - loss: 0.7936 - val_accuracy: 1.0000 - val_loss: 0.1613 - learning_rate: 0.0010
Epoch 2/12
113/114 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.9573 - loss: 0.2254
Epoch 2: val_loss improved from 0.16132 to 0.08240, saving model to /content/drive/MyDrive/underwater_plastics/models/best_model_20251102-065407.h5


114/114 ━━━━━━━━━━━━━━━━━━━━ 11s 100ms/step - accuracy: 0.9576 - loss: 0.2245 - val_accuracy: 1.0000 - val_loss: 0.0824 - learning_rate: 0.0010
Epoch 3/12
113/114 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.9988 - loss: 0.0740
Epoch 3: val_loss improved from 0.08240 to 0.04212, saving model to /content/drive/MyDrive/underwater_plastics/models/best_model_20251102-065407.h5


114/114 ━━━━━━━━━━━━━━━━━━━━ 11s 94ms/step - accuracy: 0.9988 - loss: 0.0738 - val_accuracy: 1.0000 - val_loss: 0.0421 - learning_rate: 0.0010
Epoch 4/12
113/114 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.9999 - loss: 0.0415
Epoch 4: val_loss improved from 0.04212 to 0.03787, saving model to /content/drive/MyDrive/underwater_plastics/models/best_model_20251102-065407.h5


114/114 ━━━━━━━━━━━━━━━━━━━━ 11s 96ms/step - accuracy: 0.9999 - loss: 0.0414 - val_accuracy: 1.0000 - val_loss: 0.0379 - learning_rate: 0.0010
Epoch 5/12
113/114 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 1.0000 - loss: 0.0325
Epoch 5: val_loss improved from 0.03787 to 0.02769, saving model to /content/drive/MyDrive/underwater_plastics/models/best_model_20251102-065407.h5


...

114/114 ━━━━━━━━━━━━━━━━━━━━ 11s 95ms/step - accuracy: 1.0000 - loss: 0.0239 - val_accuracy: 1.0000 - val_loss: 0.0222 - learning_rate: 0.0010
Epoch 9/12
113/114 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 1.0000 - loss: 0.0224
Epoch 9: val_loss improved from 0.02222 to 0.02103, saving model to /content/drive/MyDrive/underwater_plastics/models/best_model_20251102-065407.h5


114/114 ━━━━━━━━━━━━━━━━━━━━ 11s 96ms/step - accuracy: 1.0000 - loss: 0.0224 - val_accuracy: 1.0000 - val_loss: 0.0210 - learning_rate: 0.0010
Epoch 10/12
113/114 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 1.0000 - loss: 0.0212
Epoch 10: val_loss improved from 0.02103 to 0.02007, saving model to /content/drive/MyDrive/underwater_plastics/models/best_model_20251102-065407.h5


114/114 ━━━━━━━━━━━━━━━━━━━━ 11s 99ms/step - accuracy: 1.0000 - loss: 0.0212 - val_accuracy: 1.0000 - val_loss: 0.0201 - learning_rate: 0.0010
Epoch 11/12
113/114 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 1.0000 - loss: 0.0201
Epoch 11: val_loss improved from 0.02007 to 0.01909, saving model to /content/drive/MyDrive/underwater_plastics/models/best_model_20251102-065407.h5


114/114 ━━━━━━━━━━━━━━━━━━━━ 11s 94ms/step - accuracy: 1.0000 - loss: 0.0201 - val_accuracy: 1.0000 - val_loss: 0.0191 - learning_rate: 0.0010
Epoch 12/12
113/114 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 1.0000 - loss: 0.0191
Epoch 12: val_loss improved from 0.01909 to 0.01810, saving model to /content/drive/MyDrive/underwater_plastics/models/best_model_20251102-065407.h5


114/114 ━━━━━━━━━━━━━━━━━━━━ 11s 96ms/step - accuracy: 1.0000 - loss: 0.0191 - val_accuracy: 1.0000 - val_loss: 0.0181 - learning_rate: 0.0010
Restoring model weights from the end of the best epoch: 12.


In [12]:
# ===== Cell 9: Fine-tune (unfreeze top of base) =====
base_model.trainable = True
# Freeze first N layers to preserve low-level features; tune this number if needed
freeze_until = 100
for layer in base_model.layers[:freeze_until]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

EPOCHS_FINETUNE = 15
history2 = model.fit(
    train_ds,
    epochs=EPOCHS_FINETUNE,
    validation_data=val_ds,
    callbacks=[cb_early, cb_plateau, cb_ckpt]
)


Epoch 1/15
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 151ms/step - accuracy: 1.0000 - loss: 0.0189
Epoch 1: val_loss did not improve from 0.01810
114/114 ━━━━━━━━━━━━━━━━━━━━ 49s 195ms/step - accuracy: 1.0000 - loss: 0.0189 - val_accuracy: 1.0000 - val_loss: 0.0181 - learning_rate: 1.0000e-04
Epoch 2/15
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - accuracy: 1.0000 - loss: 0.0178
Epoch 2: val_loss improved from 0.01810 to 0.01663, saving model to /content/drive/MyDrive/underwater_plastics/models/best_model_20251102-065407.h5


114/114 ━━━━━━━━━━━━━━━━━━━━ 21s 183ms/step - accuracy: 1.0000 - loss: 0.0178 - val_accuracy: 1.0000 - val_loss: 0.0166 - learning_rate: 1.0000e-04
Epoch 3/15
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 153ms/step - accuracy: 1.0000 - loss: 0.0169
Epoch 3: val_loss improved from 0.01663 to 0.01617, saving model to /content/drive/MyDrive/underwater_plastics/models/best_model_20251102-065407.h5


114/114 ━━━━━━━━━━━━━━━━━━━━ 22s 193ms/step - accuracy: 1.0000 - loss: 0.0169 - val_accuracy: 1.0000 - val_loss: 0.0162 - learning_rate: 1.0000e-04
Epoch 4/15
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step - accuracy: 1.0000 - loss: 0.0161
Epoch 4: val_loss improved from 0.01617 to 0.01542, saving model to /content/drive/MyDrive/underwater_plastics/models/best_model_20251102-065407.h5


114/114 ━━━━━━━━━━━━━━━━━━━━ 22s 192ms/step - accuracy: 1.0000 - loss: 0.0161 - val_accuracy: 1.0000 - val_loss: 0.0154 - learning_rate: 1.0000e-04
Epoch 5/15
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step - accuracy: 1.0000 - loss: 0.0154
Epoch 5: val_loss improved from 0.01542 to 0.01471, saving model to /content/drive/MyDrive/underwater_plastics/models/best_model_20251102-065407.h5


114/114 ━━━━━━━━━━━━━━━━━━━━ 22s 192ms/step - accuracy: 1.0000 - loss: 0.0154 - val_accuracy: 1.0000 - val_loss: 0.0147 - learning_rate: 1.0000e-04
Epoch 6/15
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step - accuracy: 1.0000 - loss: 0.0146
Epoch 6: val_loss improved from 0.01471 to 0.01401, saving model to /content/drive/MyDrive/underwater_plastics/models/best_model_20251102-065407.h5


...

114/114 ━━━━━━━━━━━━━━━━━━━━ 22s 195ms/step - accuracy: 1.0000 - loss: 0.0109 - val_accuracy: 1.0000 - val_loss: 0.0103 - learning_rate: 1.0000e-04
Epoch 12/15
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 158ms/step - accuracy: 1.0000 - loss: 0.0102
Epoch 12: val_loss improved from 0.01032 to 0.00964, saving model to /content/drive/MyDrive/underwater_plastics/models/best_model_20251102-065407.h5


114/114 ━━━━━━━━━━━━━━━━━━━━ 23s 199ms/step - accuracy: 1.0000 - loss: 0.0102 - val_accuracy: 1.0000 - val_loss: 0.0096 - learning_rate: 1.0000e-04
Epoch 13/15
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step - accuracy: 1.0000 - loss: 0.0095
Epoch 13: val_loss improved from 0.00964 to 0.00897, saving model to /content/drive/MyDrive/underwater_plastics/models/best_model_20251102-065407.h5


114/114 ━━━━━━━━━━━━━━━━━━━━ 23s 197ms/step - accuracy: 1.0000 - loss: 0.0095 - val_accuracy: 1.0000 - val_loss: 0.0090 - learning_rate: 1.0000e-04
Epoch 14/15
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 157ms/step - accuracy: 1.0000 - loss: 0.0088
Epoch 14: val_loss improved from 0.00897 to 0.00831, saving model to /content/drive/MyDrive/underwater_plastics/models/best_model_20251102-065407.h5


114/114 ━━━━━━━━━━━━━━━━━━━━ 23s 199ms/step - accuracy: 1.0000 - loss: 0.0088 - val_accuracy: 1.0000 - val_loss: 0.0083 - learning_rate: 1.0000e-04
Epoch 15/15
114/114 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step - accuracy: 1.0000 - loss: 0.0082
Epoch 15: val_loss improved from 0.00831 to 0.00767, saving model to /content/drive/MyDrive/underwater_plastics/models/best_model_20251102-065407.h5


114/114 ━━━━━━━━━━━━━━━━━━━━ 22s 195ms/step - accuracy: 1.0000 - loss: 0.0082 - val_accuracy: 1.0000 - val_loss: 0.0077 - learning_rate: 1.0000e-04
Restoring model weights from the end of the best epoch: 15.


In [15]:
# ===== Safe Cell 10 replacement: Save & evaluate with robust error handling =====
import os, shutil, traceback
LOCAL_SAVE_DIR = f"/content/final_saved_model_{TIMESTAMP}"
DRIVE_SAVE_DIR = FINAL_MODEL_DIR  # from earlier cells
os.makedirs(LOCAL_SAVE_DIR, exist_ok=True)

try:
    # Save locally first (SavedModel format)
    print("Saving model to local path:", LOCAL_SAVE_DIR)
    model.save(LOCAL_SAVE_DIR, include_optimizer=False)
    print("Local save OK.")

    # Copy to Drive (safer than saving directly to Drive)
    try:
        if os.path.exists(DRIVE_SAVE_DIR):
            print("Drive save dir already exists. Removing it to avoid copy issues:", DRIVE_SAVE_DIR)
            shutil.rmtree(DRIVE_SAVE_DIR)
        print("Copying model to Drive at:", DRIVE_SAVE_DIR)
        shutil.copytree(LOCAL_SAVE_DIR, DRIVE_SAVE_DIR)
        print("Model copied to Drive successfully.")
    except Exception as e_copy:
        print("Failed to copy to Drive. You can manually copy from", LOCAL_SAVE_DIR)
        print("Copy error:", e_copy)

    # Evaluate on test dataset if present
    if 'test_ds' in globals() and test_ds is not None:
        print("Evaluating on test dataset...")
        eval_res = model.evaluate(test_ds)
        try:
            loss, acc = eval_res[0], eval_res[1]
            print(f"Test loss: {loss:.4f}, Test accuracy: {acc:.4f}")
        except Exception:
            # If metrics differ, just print returned evaluation results
            print("Evaluation returned:", eval_res)
    else:
        print("No test dataset available. Skipping evaluation.")

    print("Done. Model saved locally and (attempted) copied to Drive.")
    print("Local model path:", LOCAL_SAVE_DIR)
    print("Drive model path:", DRIVE_SAVE_DIR)

except Exception as e:
    print("An exception occurred while saving/evaluating the model.")
    traceback.print_exc()
    # give a simple fallback: save weights only
    try:
        weights_path = os.path.join(BASE, f"model_weights_{TIMESTAMP}.h5")
        model.save_weights(weights_path)
        print("Saved model weights as fallback to:", weights_path)
    except Exception as e_weights:
        print("Also failed to save weights. Final error:")
        traceback.print_exc()



Saving model to local path: /content/final_saved_model_20251102-065407
An exception occurred while saving/evaluating the model.
Also failed to save weights. Final error:


Traceback (most recent call last):
  File "/tmp/ipython-input-4097096938.py", line 10, in <cell line: 0>
    model.save(LOCAL_SAVE_DIR, include_optimizer=False)
  File "/usr/local/lib/python3.12/dist-packages/keras/src/utils/traceback_utils.py", line 122, in error_handler
    raise e.with_traceback(filtered_tb) from None
  File "/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_api.py", line 114, in save_model
    raise ValueError(
ValueError: Invalid filepath extension for saving. Please add either a `.keras` extension for the native Keras format (recommended) or a `.h5` extension. Use `model.export(filepath)` if you want to export a SavedModel for use with TFLite/TFServing/etc. Received: filepath=/content/final_saved_model_20251102-065407.
Traceback (most recent call last):
  File "/tmp/ipython-input-4097096938.py", line 48, in <cell line: 0>
    model.save_weights(weights_path)
  File "/usr/local/lib/python3.12/dist-packages/keras/src/utils/traceback_utils.py", line 12

In [16]:
# ===== Cell 11: Simple inference helper =====
CLASS_NAMES = CLASS_NAMES  # keep for UI

def predict_image_from_pil(pil_img):
    pil_img = pil_img.resize(IMG_SIZE)
    arr = keras_image.img_to_array(pil_img)
    arr = np.expand_dims(arr, axis=0)
    arr = preprocess_input(arr)
    preds = model.predict(arr)[0]
    # return list of (label, prob) sorted desc
    pairs = list(zip(CLASS_NAMES, preds.tolist()))
    pairs_sorted = sorted(pairs, key=lambda x: -x[1])
    return {label: float(np.round(prob, 4)) for label, prob in pairs_sorted}


In [20]:
# ======= Final Working Cell 12: Professional Gradio UI (Gradio 4.x compatible) =======
import gradio as gr
import traceback

# Ensure the helper exists
try:
    predict_image_from_pil
except NameError:
    raise RuntimeError("Run the inference cell defining 'predict_image_from_pil' first.")

TITLE = "🌊 Ocean Plastic Waste Classifier"
DESCRIPTION = (
    f"Upload an image of floating waste. The model classifies it into: {', '.join(CLASS_NAMES)}."
    "\n\n💡 Tip: Clear ocean surface photos give best results."
)

# --- Styling ---
CSS = """
.gradio-container { background: linear-gradient(180deg,#e8faff 0%, #f6fff7 100%); font-family: 'Inter', sans-serif; }
h1, h3 { color: #005f73; }
button { background-color:#0a9396 !important; color:white !important; border-radius:10px !important; padding:8px 14px !important; }
footer {display:none;}
"""

def gr_interface(img):
    """Predict uploaded image & format outputs correctly for Gradio 4.x"""
    try:
        if img is None:
            return {"Error": 1.0}, gr.BarPlot.update(value={}, x=[], y=[])

        preds = predict_image_from_pil(img)
        preds = {k: float(v) for k, v in preds.items()}

        # Prepare data for bar chart (Gradio 4.x)
        x = list(preds.keys())
        y = list(preds.values())
        bar_output = gr.BarPlot.update(value={"x": x, "y": y}, x=x, y=y)

        return preds, bar_output

    except Exception as e:
        print("Error during prediction:", e)
        traceback.print_exc()
        return {"Error": 1.0}, gr.BarPlot.update(value={}, x=[], y=[])

with gr.Blocks(css=CSS, title=TITLE, theme=gr.themes.Soft()) as demo:
    gr.Markdown(f"# {TITLE}")
    gr.Markdown(DESCRIPTION)
    with gr.Row():
        with gr.Column(scale=1):
            inp = gr.Image(type="pil", label="📸 Upload image (jpg/png)")
            btn = gr.Button("🔍 Classify Waste")
            st = gr.Markdown("**ℹ️ Model stored in Drive under `underwater_plastics/models/`.**")
        with gr.Column(scale=1):
            out_label = gr.Label(label="Predicted Categories")
            out_bar = gr.BarPlot(label="Confidence Distribution")
    btn.click(gr_interface, inputs=inp, outputs=[out_label, out_bar])

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9620bce3b7cdc55ccc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
